In [ ]:
import pandas as pd
import requests

base_url= "https://api.census.gov/data/2021/acs/acs5"


In [ ]:
variables = [
    "NAME",
    "B19013_001E",
    "B02001_001E", "B02001_002E",
    "B25003_001E", "B25003_003E",
    "B25010_001E",
    "B25070_001E", "B25070_007E", "B25070_008E", "B25070_009E", "B25070_010E",

    # children from B01001
    "B01001_003E", "B01001_004E", "B01001_005E", "B01001_006E",
    "B01001_027E", "B01001_028E", "B01001_029E", "B01001_030E"
]

In [ ]:
#manhattan only
params = {
    "get": ",".join(variables),
    "for": "tract:*",
    "in": "state:36 county:061"   # Manhattan
}

response = requests.get(base_url, params=params)
data = response.json()

df_manhattan = pd.DataFrame(data[1:], columns=data[0])
df_manhattan.head()

,NAME,B19013_001E,B02001_001E,B02001_002E,B25003_001E,B25003_003E,B25010_001E,B25070_001E,B25070_007E,B25070_008E,...,B01001_004E,B01001_005E,B01001_006E,B01001_027E,B01001_028E,B01001_029E,B01001_030E,state,county,tract
0,"Census Tract 1, New York County, New York",-666666666,0,0,0,0,-666666666.00,0,0,0,...,0,0,0,0,0,0,0,36,061,000100
1,"Census Tract 2.01, New York County, New York",32286,2740,347,856,856,3.13,856,76,33,...,421,357,88,20,64,226,0,36,061,000201
2,"Census Tract 2.02, New York County, New York",31806,6789,2807,3235,2320,2.01,2320,417,213,...,91,285,48,120,194,118,68,36,061,000202
3,"Census Tract 5, New York County, New York",-666666666,0,0,0,0,-666666666.00,0,0,0,...,0,0,0,0,0,0,0,36,061,000500
4,"Census Tract 6, New York County, New York",19692,10732,1053,4779,4613,2.21,4613,649,215,...,155,284,77,179,178,161,83,36,061,000600


In [ ]:
nyc_counties = ["005", "047", "061", "081", "085"]

dfs = []

for county in nyc_counties:
    params = {
        "get": ",".join(variables),
        "for": "tract:*",
        "in": f"state:36 county:{county}"
    }

    response = requests.get(base_url, params=params)
    data = response.json()
    temp = pd.DataFrame(data[1:], columns=data[0])
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)
df.head()

,NAME,B19013_001E,B02001_001E,B02001_002E,B25003_001E,B25003_003E,B25010_001E,B25070_001E,B25070_007E,B25070_008E,...,B01001_004E,B01001_005E,B01001_006E,B01001_027E,B01001_028E,B01001_029E,B01001_030E,state,county,tract
0,"Census Tract 1, Bronx County, New York",-666666666,6661,2680,0,0,-666666666.00,0,0,0,...,0,0,98,0,0,0,0,36,005,000100
1,"Census Tract 2, Bronx County, New York",70867,4453,1235,1392,628,3.20,628,28,15,...,144,202,85,91,114,208,14,36,005,000200
2,"Census Tract 4, Bronx County, New York",98090,6000,1809,2199,694,2.72,694,38,98,...,195,129,116,51,129,75,332,36,005,000400
3,"Census Tract 16, Bronx County, New York",40033,6038,996,2187,1831,2.66,1831,495,98,...,159,129,127,49,149,205,249,36,005,001600
4,"Census Tract 19.01, Bronx County, New York",55924,2168,664,885,885,2.43,885,81,17,...,37,45,29,79,68,127,12,36,005,001901


In [ ]:
df["GEOID"] = df["state"] + df["county"] + df["tract"]
df[["NAME", "state", "county", "tract", "GEOID"]].head()

,NAME,state,county,tract,GEOID
0,"Census Tract 1, Bronx County, New York",36,005,000100,36005000100
1,"Census Tract 2, Bronx County, New York",36,005,000200,36005000200
2,"Census Tract 4, Bronx County, New York",36,005,000400,36005000400
3,"Census Tract 16, Bronx County, New York",36,005,001600,36005001600
4,"Census Tract 19.01, Bronx County, New York",36,005,001901,36005001901


In [ ]:
num_cols = [col for col in df.columns if col not in ["NAME", "state", "county", "tract", "GEOID"]]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
df["median_income"] = df["B19013_001E"]

df["pct_nonwhite"] = 1 - (df["B02001_002E"] / df["B02001_001E"])

df["pct_renter"] = df["B25003_003E"] / df["B25003_001E"]

df["avg_hh_size"] = df["B25010_001E"]

df["children_count"] = (
    df["B01001_003E"] + df["B01001_004E"] + df["B01001_005E"] + df["B01001_006E"] +
    df["B01001_027E"] + df["B01001_028E"] + df["B01001_029E"] + df["B01001_030E"]
)

df["rent_burdened_count"] = (
    df["B25070_007E"] + df["B25070_008E"] + df["B25070_009E"] + df["B25070_010E"]
)

df["pct_rent_burdened"] = df["rent_burdened_count"] / df["B25070_001E"]

In [ ]:
final_df = df[
    [
        "GEOID", "NAME",
        "median_income",
        "pct_nonwhite",
        "pct_renter",
        "avg_hh_size",
        "children_count",
        "pct_rent_burdened", "tract"
    ]
].copy()

final_df.head()

,GEOID,NAME,median_income,pct_nonwhite,pct_renter,avg_hh_size,children_count,pct_rent_burdened,tract
0,36005000100,"Census Tract 1, Bronx County, New York",-666666666,0.597658,NaN,-6.666667e+08,98,NaN,000100
1,36005000200,"Census Tract 2, Bronx County, New York",70867,0.722659,0.451149,3.200000e+00,1008,0.531847,000200
2,36005000400,"Census Tract 4, Bronx County, New York",98090,0.698500,0.315598,2.720000e+00,1156,0.465418,000400
3,36005001600,"Census Tract 16, Bronx County, New York",40033,0.835045,0.837220,2.660000e+00,1281,0.566357,001600
4,36005001901,"Census Tract 19.01, Bronx County, New York",55924,0.693727,1.000000,2.430000e+00,506,0.549153,001901


In [ ]:
final_df.shape
final_df.isna().sum()
final_df.head()


,GEOID,NAME,median_income,pct_nonwhite,pct_renter,avg_hh_size,children_count,pct_rent_burdened,tract
0,36005000100,"Census Tract 1, Bronx County, New York",-666666666,0.597658,NaN,-6.666667e+08,98,NaN,000100
1,36005000200,"Census Tract 2, Bronx County, New York",70867,0.722659,0.451149,3.200000e+00,1008,0.531847,000200
2,36005000400,"Census Tract 4, Bronx County, New York",98090,0.698500,0.315598,2.720000e+00,1156,0.465418,000400
3,36005001600,"Census Tract 16, Bronx County, New York",40033,0.835045,0.837220,2.660000e+00,1281,0.566357,001600
4,36005001901,"Census Tract 19.01, Bronx County, New York",55924,0.693727,1.000000,2.430000e+00,506,0.549153,001901


In [ ]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2327 entries, 0 to 2326
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   GEOID              2327 non-null   object 
 1   NAME               2327 non-null   object 
 2   median_income      2327 non-null   int64  
 3   pct_nonwhite       2239 non-null   float64
 4   pct_renter         2229 non-null   float64
 5   avg_hh_size        2327 non-null   float64
 6   children_count     2327 non-null   int64  
 7   pct_rent_burdened  2221 non-null   float64
 8   tract              2327 non-null   object 
dtypes: float64(4), int64(2), object(3)
memory usage: 163.7+ KB


In [ ]:
final_df.isna().sum().sort_values(ascending=False)

,0
pct_rent_burdened,106
pct_renter,98
pct_nonwhite,88
GEOID,0
median_income,0
NAME,0
avg_hh_size,0
children_count,0
tract,0


In [ ]:
final_df = final_df.dropna()

In [ ]:
final_df.isna().sum()

,0
GEOID,0
NAME,0
median_income,0
pct_nonwhite,0
pct_renter,0
avg_hh_size,0
children_count,0
pct_rent_burdened,0
tract,0


In [ ]:
final_df.shape

(2221, 9)

In [ ]:
final_df.head(

)

,GEOID,NAME,median_income,pct_nonwhite,pct_renter,avg_hh_size,children_count,pct_rent_burdened,tract
1,36005000200,"Census Tract 2, Bronx County, New York",70867,0.722659,0.451149,3.20,1008,0.531847,000200
2,36005000400,"Census Tract 4, Bronx County, New York",98090,0.698500,0.315598,2.72,1156,0.465418,000400
3,36005001600,"Census Tract 16, Bronx County, New York",40033,0.835045,0.837220,2.66,1281,0.566357,001600
4,36005001901,"Census Tract 19.01, Bronx County, New York",55924,0.693727,1.000000,2.43,506,0.549153,001901
5,36005001902,"Census Tract 19.02, Bronx County, New York",60804,0.791994,0.792553,2.72,397,0.701342,001902


In [ ]:
final_df.to_csv("clean_census_data.csv", index=False)